<a href="https://colab.research.google.com/github/truonghoanganh2709/bigdata/blob/main/Hadoop_Lab_Ecommerce_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phần 0. Cài đặt và đọc dữ liệu

In [2]:
import os, shutil, glob, subprocess, csv
from pathlib import Path
from collections import Counter, defaultdict

CSV = "orders_2026_09.csv"          # đặt cùng thư mục với notebook
HDFS_BASE = "/user/student/ecommerce"
HDFS_IN   = f"{HDFS_BASE}/input"
HDFS_OUT  = f"{HDFS_BASE}/output/revenue_by_category"

with open(CSV, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))          # mỗi dòng là một dict {tên cột: giá trị}

print("Số dòng:", len(rows), "| Số cột:", len(rows[0]))
print("Các cột:", list(rows[0].keys()))
for r in rows[:3]:
    print(r)

Số dòng: 1500 | Số cột: 11
Các cột: ['transaction_id', 'order_date', 'order_time', 'province', 'category', 'channel', 'quantity', 'unit_price', 'amount', 'status', 'shipping_days']
{'transaction_id': 'TX000001', 'order_date': '9/30/2026', 'order_time': '2:25:00', 'province': 'Ha Noi', 'category': 'Beauty', 'channel': 'web', 'quantity': '1', 'unit_price': '294947', 'amount': '294947', 'status': 'SUCCESS', 'shipping_days': '2'}
{'transaction_id': 'TX000002', 'order_date': '9/23/2026', 'order_time': '23:58:00', 'province': 'Da Nang', 'category': 'Grocery', 'channel': 'mobile', 'quantity': '1', 'unit_price': '265514', 'amount': '265514', 'status': 'SUCCESS', 'shipping_days': '3'}
{'transaction_id': 'TX000003', 'order_date': '9/5/2026', 'order_time': '15:38:00', 'province': 'Ha Noi', 'category': 'Electronics', 'channel': 'partner', 'quantity': '1', 'unit_price': '12741238', 'amount': '12741238', 'status': 'SUCCESS', 'shipping_days': '3'}


In [3]:
def show_counter(title, c):
    print(title)
    for k, v in c.most_common():
        print(f"  {k:<14}{v:>6,}")

show_counter("Trạng thái đơn hàng:", Counter(r["status"].strip() for r in rows))
show_counter("\nSố đơn theo category:", Counter(r["category"].strip() for r in rows))
missing = sum(1 for r in rows for v in r.values() if v is None or v.strip() == "")
print("\nÔ dữ liệu thiếu:", missing)

Trạng thái đơn hàng:
  SUCCESS        1,240
  FAILED           142
  RETURNED         118

Số đơn theo category:
  Electronics      353
  Grocery          282
  Fashion          281
  Beauty           145
  Home             142
  Book             114
  Sport             96
  Toy               87

Ô dữ liệu thiếu: 0


## Phần 1. Mapper và Reducer
- **Mapper:** đọc từng dòng CSV; nếu `status = SUCCESS` thì in ra cặp `category<TAB>amount`.
- **Shuffle/Sort:** Hadoop gom các bản ghi cùng `category` về cùng một reducer (đã sắp theo key).
- **Reducer:** cộng dồn `amount` theo từng `category` và ghi kết quả cuối.

In [4]:
MAPPER = r"""#!/usr/bin/env python3
import sys, csv

# Cot: transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
for row in csv.reader(sys.stdin):
    if len(row) < 10 or row[0] == "transaction_id":   # bo header / dong loi
        continue
    category = row[4].strip()
    status = row[9].strip().upper()
    try:
        amount = int(float(row[8].replace(",", "").strip()))
    except ValueError:
        continue
    if status == "SUCCESS":
        print(f"{category}\t{amount}")                # key = category, value = amount
"""

REDUCER = r"""#!/usr/bin/env python3
import sys

current_key, total = None, 0
for line in sys.stdin:
    key, _, value = line.rstrip("\n").partition("\t")
    try:
        value = int(value)
    except ValueError:
        continue
    if key == current_key:
        total += value
    else:
        if current_key is not None:
            print(f"{current_key}\t{total}")
        current_key, total = key, value
if current_key is not None:
    print(f"{current_key}\t{total}")
"""

Path("mapper_revenue.py").write_text(MAPPER, encoding="utf-8")
Path("reducer_revenue.py").write_text(REDUCER, encoding="utf-8")
for f in ("mapper_revenue.py", "reducer_revenue.py"):
    os.chmod(f, 0o755)
print("Đã ghi mapper_revenue.py và reducer_revenue.py")

Đã ghi mapper_revenue.py và reducer_revenue.py


## Phần 2. Mô phỏng MapReduce ở local (map → sort → reduce)
Chỉ để kiểm tra logic trước khi đưa lên Hadoop.

In [5]:
import sys

def sh(cmd, show=True):
    """Chạy lệnh shell (dùng cho các lệnh hadoop/yarn), in stdout+stderr, trả về stdout."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    if show: print(out.rstrip())
    return p.stdout

def run_py(script, text):
    """Chạy 1 script Python bằng ĐÚNG interpreter của notebook, đưa `text` vào stdin."""
    p = subprocess.run([sys.executable, script], input=text, capture_output=True,
                       text=True, encoding="utf-8")
    if p.returncode != 0 or p.stderr.strip():
        print(f"[CẢNH BÁO] {script}: returncode={p.returncode}\n{p.stderr.strip()}")
    return p.stdout

raw = Path(CSV).read_text(encoding="utf-8-sig")        # utf-8-sig: bỏ ký tự BOM nếu có

# 1) MAP
mapped_out = run_py("mapper_revenue.py", raw)
print("== 5 cặp key-value đầu tiên do Mapper phát ra ==")
print(mapped_out.splitlines()[:5])

# 2) SHUFFLE / SORT: sắp xếp theo key để các bản ghi cùng category nằm cạnh nhau
shuffled = "\n".join(sorted(mapped_out.splitlines())) + "\n"

# 3) REDUCE
local_out = run_py("reducer_revenue.py", shuffled)
Path("result_local.txt").write_text(local_out, encoding="utf-8")
print("\n== Kết quả mô phỏng local (key<TAB>tổng doanh thu) ==")
print(local_out)

== 5 cặp key-value đầu tiên do Mapper phát ra ==
['Beauty\t294947', 'Grocery\t265514', 'Electronics\t12741238', 'Electronics\t8846997', 'Beauty\t839161']

== Kết quả mô phỏng local (key<TAB>tổng doanh thu) ==
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779



In [6]:
# Đối chiếu độc lập: cộng trực tiếp bằng Python (không qua mapper/reducer)
check = defaultdict(int)
n_success = 0
for r in rows:
    if r["status"].strip().upper() == "SUCCESS":
        n_success += 1
        check[r["category"].strip()] += int(float(r["amount"].replace(",", "")))

local = {}
for line in local_out.splitlines():
    if "\t" in line:
        k, v = line.split("\t"); local[k] = int(v)

ok = (dict(check) == local)
if ok:
    print("✅ Kết quả MapReduce local khớp tính trực tiếp.")
else:
    print("❌ Kết quả KHÁC NHAU. Thông tin để chẩn đoán:")
    print(f"  Số đơn SUCCESS đếm trực tiếp : {n_success}")
    print(f"  Số cặp key-value Mapper phát : {len(mapped_out.splitlines())}  (phải bằng số đơn SUCCESS)")
    print(f"  Số category: trực tiếp={len(check)} | MapReduce local={len(local)}")
    for k in sorted(set(check) | set(local)):
        if check.get(k) != local.get(k):
            print(f"  - {k!r:<16} trực tiếp={check.get(k)}  mapreduce={local.get(k)}")
    print("  Đầu ra Mapper (3 dòng):", mapped_out.splitlines()[:3])
    print("  Đầu ra Reducer (thô)  :", repr(local_out[:200]))

✅ Kết quả MapReduce local khớp tính trực tiếp.


## Phần 3. Chạy thật trên HDFS và Hadoop Streaming
Chạy phần này **trong môi trường lab có Hadoop** (LabEx/RunCode...). Nếu máy không có Hadoop, các ô dưới sẽ báo và bỏ qua, không gây lỗi.

📸 Chụp màn hình output của các ô 3.2 → 3.5 để làm minh chứng.

### 3.0 Tìm hoặc cài Hadoop
Ô dưới đây (1) dùng `hadoop` nếu đã có, (2) tìm Hadoop cài sẵn nhưng chưa nằm trong `PATH`, (3) nếu vẫn không có và `AUTO_INSTALL = True` thì **tải và cài Hadoop 3.3.6** (cần Internet, hơn 600 MB, mất vài phút) rồi bật HDFS 1 node ngay trong máy chạy notebook.  
Không muốn cài tự động thì đặt `AUTO_INSTALL = False`.

In [ ]:
# 3.0  Tìm hoặc cài Hadoop (chỉ làm khi lệnh `hadoop` chưa dùng được)
import shutil, glob, os, subprocess, getpass, urllib.request
from pathlib import Path

AUTO_INSTALL   = True      # True: nếu máy chưa có Hadoop thì tải về và cài (HDFS 1 node, chạy trong notebook)
HADOOP_VERSION = "3.3.6"

def _run(cmd, timeout=1800):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, (p.stdout or "") + (p.stderr or "")

def _add_to_path(hadoop_home):
    os.environ["HADOOP_HOME"] = hadoop_home
    for d in (f"{hadoop_home}/bin", f"{hadoop_home}/sbin"):
        if d not in os.environ["PATH"].split(os.pathsep):
            os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]

def find_existing_hadoop():
    """Tìm Hadoop đã cài sẵn nhưng chưa nằm trong PATH."""
    h = os.environ.get("HADOOP_HOME")
    if h and Path(h, "bin/hadoop").exists():
        return h
    for pat in ["/usr/local/hadoop*", "/opt/hadoop*", "/usr/lib/hadoop*", "/usr/hdp/*/hadoop",
                str(Path.home() / "hadoop*"), "/home/*/hadoop*", "/opt/*/hadoop*"]:
        for h in sorted(glob.glob(pat)):
            if Path(h, "bin/hadoop").exists():
                return h
    return None

def find_java_home():
    jh = os.environ.get("JAVA_HOME")
    if jh and Path(jh, "bin/java").exists():
        return jh
    j = shutil.which("java")
    if j:
        return str(Path(os.path.realpath(j)).parent.parent)
    for h in sorted(glob.glob("/usr/lib/jvm/*")):
        if Path(h, "bin/java").exists():
            return h
    return None

def install_hadoop():
    java_home = find_java_home()
    if not java_home:
        print("Chưa có Java -> cài openjdk-11 (cần quyền root, khoảng 1-2 phút)...")
        rc, out = _run("apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless", timeout=900)
        java_home = find_java_home()
        if not java_home:
            print("Không cài được Java:", out[-500:]); return None
    os.environ["JAVA_HOME"] = java_home

    home = Path.home() / f"hadoop-{HADOOP_VERSION}"
    if not Path(home, "bin/hadoop").exists():
        tgz = Path.home() / f"hadoop-{HADOOP_VERSION}.tar.gz"
        url = f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz"
        if not tgz.exists():
            print("Đang tải Hadoop (hơn 600 MB, mất vài phút)...")
            try:
                urllib.request.urlretrieve(url, str(tgz))
            except Exception as e:
                print("Tải thất bại:", e)
                if tgz.exists(): tgz.unlink()
                return None
        rc, out = _run(f"tar -xzf {tgz} -C {Path.home()}")
        if rc:
            print("Giải nén lỗi:", out[-500:]); return None
    return str(home)

def configure_and_start_hdfs(home, java_home):
    conf, data = Path(home, "etc/hadoop"), Path.home() / "hdfs_data"
    (conf / "core-site.xml").write_text(
        '<?xml version="1.0"?>\n<configuration>\n'
        '  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>\n'
        '</configuration>\n')
    (conf / "hdfs-site.xml").write_text(
        '<?xml version="1.0"?>\n<configuration>\n'
        '  <property><name>dfs.replication</name><value>1</value></property>\n'
        f'  <property><name>dfs.namenode.name.dir</name><value>file://{data}/nn</value></property>\n'
        f'  <property><name>dfs.datanode.data.dir</name><value>file://{data}/dn</value></property>\n'
        '</configuration>\n')
    env_sh = conf / "hadoop-env.sh"
    if f"export JAVA_HOME={java_home}" not in env_sh.read_text():
        with open(env_sh, "a") as f:
            f.write(f"\nexport JAVA_HOME={java_home}\n")

    user = "root" if os.geteuid() == 0 else getpass.getuser()
    for v in ("HDFS_NAMENODE_USER", "HDFS_DATANODE_USER", "HDFS_SECONDARYNAMENODE_USER"):
        os.environ[v] = user

    if not (data / "nn" / "current").exists():
        print("Định dạng (format) NameNode lần đầu...")
        rc, out = _run("hdfs namenode -format -force -nonInteractive", timeout=300)
        if rc: print(out[-800:])
    print("Khởi động HDFS (NameNode + DataNode)...")
    for d in ("namenode", "datanode"):
        _run(f"hdfs --daemon start {d}", timeout=120)
    _run("hdfs dfsadmin -safemode wait", timeout=180)

if shutil.which("hadoop"):
    print("Đã có lệnh hadoop:", shutil.which("hadoop"))
else:
    found = find_existing_hadoop()
    if found:
        print("Tìm thấy Hadoop có sẵn tại:", found, "-> thêm vào PATH")
        _add_to_path(found)
        os.environ.setdefault("JAVA_HOME", find_java_home() or "")
    elif AUTO_INSTALL:
        print("Máy chưa có Hadoop -> thử cài tự động (cần Internet)...")
        h = install_hadoop()
        if h:
            _add_to_path(h)
            configure_and_start_hdfs(h, os.environ["JAVA_HOME"])
    else:
        print("Không có Hadoop. Đặt AUTO_INSTALL = True để cài, hoặc mở notebook trong lab có Hadoop.")

print("\nhadoop ->", shutil.which("hadoop") or "KHÔNG TÌM THẤY")
if shutil.which("hadoop"):
    print(_run("hadoop version | head -1")[1].strip())

Máy chưa có Hadoop -> thử cài tự động (cần Internet)...
Đang tải Hadoop (hơn 600 MB, mất vài phút)...


In [ ]:
HAS_HADOOP = shutil.which("hadoop") is not None
print("Có Hadoop:", HAS_HADOOP)

def hd(cmd):
    """Chạy lệnh Hadoop; bỏ qua nếu môi trường không có Hadoop."""
    if not HAS_HADOOP:
        print("[bỏ qua - không tìm thấy hadoop] $", cmd); return ""
    print("$", cmd)
    return sh(cmd)

# 3.1 Kiểm tra môi trường
hd("hadoop version | head -1")
hd("hadoop fs -ls /")
sh("python3 --version")

### 3.2 Bước 1: Đưa dữ liệu lên HDFS  

In [ ]:
hd(f"hadoop fs -mkdir -p {HDFS_IN}")
hd(f"hadoop fs -mkdir -p {HDFS_BASE}/output")
hd(f"hadoop fs -put -f {CSV} {HDFS_IN}/")
hd(f"hadoop fs -ls {HDFS_IN}")
hd(f"hadoop fs -head {HDFS_IN}/{CSV}")

### 3.3 Bước 3: Chạy Hadoop Streaming  

In [ ]:
def find_streaming_jar():
    home = os.environ.get("HADOOP_HOME", "")
    jars = glob.glob(f"{home}/share/hadoop/tools/lib/hadoop-streaming*.jar") if home else []
    if not jars and home:
        jars = glob.glob(f"{home}/**/*streaming*.jar", recursive=True)
    return jars[0] if jars else None

JAR = find_streaming_jar() if HAS_HADOOP else None
print("Streaming jar:", JAR)

if HAS_HADOOP and JAR:
    hd(f"hadoop fs -rm -r -f {HDFS_OUT}")          # xóa output cũ trước khi chạy lại
    hd(f"hadoop jar {JAR} "
       f"-files mapper_revenue.py,reducer_revenue.py "
       f"-mapper mapper_revenue.py -reducer reducer_revenue.py "
       f"-input {HDFS_IN}/{CSV} -output {HDFS_OUT}")
else:
    print("Bỏ qua: không có Hadoop/streaming jar trong môi trường này.")

### 3.4 Bước 4: Xem và tải kết quả

In [ ]:
hd(f"hadoop fs -ls {HDFS_OUT}")
hd(f"hadoop fs -cat {HDFS_OUT}/part-*")
hd(f"hadoop fs -get -f {HDFS_OUT}/part-* ./result_revenue_by_category.txt")

### 3.5 Bước 5: Quan sát YARN (nếu lab hỗ trợ)

In [ ]:
hd("yarn application -list -appStates ALL | tail -5")
# hd("yarn application -status <application_id>")
# hd("yarn logs -applicationId <application_id>")

## Phần 4. Top 3 nhóm sản phẩm, biểu đồ và bảng báo cáo

In [ ]:
# Dùng kết quả Hadoop thật nếu có, nếu không dùng kết quả mô phỏng local
src = "result_revenue_by_category.txt" if Path("result_revenue_by_category.txt").exists() else "result_local.txt"
print("Nguồn dữ liệu:", src, "(Hadoop thật)" if src.startswith("result_revenue") else "(mô phỏng local)")

res = []
for line in Path(src).read_text().splitlines():
    parts = line.split()
    if len(parts) >= 2:
        res.append((" ".join(parts[:-1]), int(parts[-1])))
res.sort(key=lambda x: -x[1])               # sắp xếp doanh thu giảm dần
total = sum(v for _, v in res)

print(f"\n{'Hạng':<6}{'Category':<14}{'Doanh thu (VND)':>20}{'Tỷ trọng':>10}")
for i, (k, v) in enumerate(res, 1):
    print(f"{i:<6}{k:<14}{v:>20,}{v/total*100:>9.1f}%")

In [ ]:
print("TOP 3 NHÓM SẢN PHẨM THEO DOANH THU (status = SUCCESS)")
for i, (k, v) in enumerate(res[:3], 1):
    print(f"  {i}. {k:<12}{v:>16,} VND  ({v/total*100:.1f}%)")
print(f"\nTổng doanh thu SUCCESS: {total:,} VND")
print(f"3 nhóm đầu chiếm {sum(v for _, v in res[:3])/total*100:.1f}% tổng doanh thu")

In [ ]:
# Biểu đồ cột Top category — vẽ bằng HTML/CSS thuần, KHÔNG cần matplotlib
TOP_COLOR, OTHER_COLOR = "#1f6feb", "#9aa4b2"
TITLE = "Doanh thu theo category - chỉ tính đơn SUCCESS (09/2026), đơn vị: tỷ VND"
mx = max(v for _, v in res)

# 1) Bản HTML để hiện ngay trong notebook
parts = ['<div style="font-family:sans-serif;max-width:720px">',
         f'<div style="font-weight:600;margin-bottom:8px">{TITLE}</div>']
for i, (k, v) in enumerate(res):
    w = max(v / mx * 100, 0.5)
    c = TOP_COLOR if i < 3 else OTHER_COLOR
    parts.append(
        '<div style="display:flex;align-items:center;margin:4px 0">'
        f'<div style="width:100px;font-size:13px">{k}</div>'
        f'<div style="flex:1;background:#eef1f5"><div style="width:{w:.1f}%;background:{c};height:22px"></div></div>'
        f'<div style="width:70px;text-align:right;font-size:13px">{v/1e9:.2f}</div></div>')
parts.append('<div style="font-size:12px;color:#666;margin-top:6px">Xanh đậm = Top 3</div></div>')
chart_html = "".join(parts)

# 2) Bản SVG lưu ra file (chèn vào báo cáo/PDF được)
BAR_H, GAP, LEFT, WIDTH = 26, 8, 110, 520
H = 40 + len(res) * (BAR_H + GAP) + 10
svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{LEFT + WIDTH + 80}" height="{H}" '
       'font-family="sans-serif" font-size="13">',
       '<rect width="100%" height="100%" fill="white"/>',
       f'<text x="0" y="20" font-weight="bold">{TITLE}</text>']
for i, (k, v) in enumerate(res):
    y = 36 + i * (BAR_H + GAP)
    w = max(v / mx * WIDTH, 2)
    c = TOP_COLOR if i < 3 else OTHER_COLOR
    svg += [f'<text x="0" y="{y + 18}">{k}</text>',
            f'<rect x="{LEFT}" y="{y}" width="{w:.1f}" height="{BAR_H}" fill="{c}"/>',
            f'<text x="{LEFT + w + 6:.1f}" y="{y + 18}">{v/1e9:.2f}</text>']
svg.append("</svg>")
Path("top_category_revenue.svg").write_text("\n".join(svg), encoding="utf-8")
print("Đã lưu top_category_revenue.svg")

# 3) Hiển thị
try:
    from IPython.display import HTML, display
    display(HTML(chart_html))
except ImportError:
    print("(Không có IPython, hiển thị dạng chữ)\n")
    for k, v in res:
        print(f"{k:<12}{'█' * max(1, int(v / mx * 40))} {v/1e9:.2f} tỷ")

# 4) Tùy chọn: nếu có matplotlib thì lưu thêm bản PNG
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar([k for k, _ in res], [v / 1e9 for _, v in res],
           color=[TOP_COLOR if i < 3 else OTHER_COLOR for i in range(len(res))])
    ax.set_ylabel("Doanh thu (tỷ VND)"); ax.set_title(TITLE, fontsize=10)
    plt.xticks(rotation=30); plt.tight_layout()
    plt.savefig("top_category_revenue.png", dpi=150); plt.close(fig)
    print("Đã lưu thêm top_category_revenue.png")
except ImportError:
    pass

## Phần 5. Phần mở rộng (tùy chọn)

### 5.1 Doanh thu theo `province` (đổi khóa từ `row[4]` sang `row[3]` trong mapper)

In [ ]:
by_prov = defaultdict(int)
for r in rows:
    if r["status"].strip().upper() == "SUCCESS":
        by_prov[r["province"].strip()] += int(float(r["amount"].replace(",", "")))
for k, v in sorted(by_prov.items(), key=lambda x: -x[1]):
    print(f"{k:<14}{v:>18,}")

### 5.2 Tỷ lệ `FAILED` theo `channel`

In [ ]:
tot, failed = Counter(), Counter()
for r in rows:
    ch = r["channel"].strip(); tot[ch] += 1
    if r["status"].strip().upper() == "FAILED": failed[ch] += 1
print(f"{'Channel':<12}{'Số đơn':>8}{'FAILED':>8}{'Tỷ lệ':>9}")
for ch in sorted(tot, key=lambda c: -failed[c] / tot[c]):
    print(f"{ch:<12}{tot[ch]:>8}{failed[ch]:>8}{failed[ch]/tot[ch]*100:>8.2f}%")

### 5.3 Số ngày giao hàng trung bình theo `province`

In [ ]:
s_sum, s_cnt = defaultdict(float), Counter()
for r in rows:
    p = r["province"].strip(); s_sum[p] += float(r["shipping_days"]); s_cnt[p] += 1
for p in sorted(s_sum, key=lambda p: s_sum[p] / s_cnt[p]):
    print(f"{p:<14}{s_sum[p]/s_cnt[p]:>6.2f} ngày")

### 5.4 Combiner: giảm dữ liệu qua shuffle
Với phép cộng, có thể dùng chính reducer làm combiner (`-combiner reducer_revenue.py`) vì phép cộng có tính kết hợp.

In [ ]:
from collections import defaultdict
mapped = mapped_out.splitlines()

# Giả lập 4 map task, mỗi task xử lý một phần dữ liệu (input split)
N_TASKS = 4
chunks = [mapped[i::N_TASKS] for i in range(N_TASKS)]

def combine(lines):
    acc = defaultdict(int)
    for ln in lines:
        k, v = ln.split("\t"); acc[k] += int(v)
    return acc

after_combiner = [combine(c) for c in chunks]          # mỗi map task gộp cục bộ trước khi shuffle
n_before = len(mapped)
n_after  = sum(len(a) for a in after_combiner)
print(f"Cặp key-value đi qua shuffle KHÔNG có combiner: {n_before:,}")
print(f"Cặp key-value đi qua shuffle CÓ combiner ({N_TASKS} map task): {n_after:,}")
print(f"Giảm ~ {100*(1-n_after/n_before):.1f}% dữ liệu truyền qua mạng")

# Kết quả cuối phải không đổi
final = defaultdict(int)
for a in after_combiner:
    for k, v in a.items(): final[k] += v
assert dict(final) == dict(check), "Combiner làm sai kết quả!"
print("Kết quả sau combiner vẫn đúng.")
# Khi chạy Hadoop: thêm  -combiner reducer_revenue.py  vào lệnh streaming